In [11]:
# using Distributions 

# abstract type Distribution{T} end
# struct Half_Cauchy <: Distribution{Float64} end
# const half_cauchy = Half_Cauchy()

# function logpdf(::Half_Cauchy, x::Real, x0::Real, gamma::Real)
#     if x<0
#             x = x*-1
#     end
            
#     return Distributions.logpdf(Distributions.Cauchy(x0, gamma), x)
    
# end

# function logpdf_grad(::Half_Cauchy, x::Real, x0::Real, gamma::Real)
#     if x<0
#             x = x*-1
#     end
            
#     x_x0 = x - x0
#     x_x0_sq = x_x0^2
#     gamma_sq = gamma^2
#     deriv_x0 =  2 * x_x0 / (gamma_sq + x_x0_sq)
#     deriv_x = - deriv_x0
#     deriv_gamma = (x_x0_sq - gamma_sq) / (gamma * (gamma_sq + x_x0_sq))
#     (deriv_x, deriv_x0, deriv_gamma)
# end

# is_discrete(::Half_Cauchy) = false

# random(::Half_Cauchy, x0::Real, gamma::Real) = abs(rand(Distributions.Cauchy(x0, gamma)))

# (::Half_Cauchy)(x0::Real, gamma::Real) = random(Half_Cauchy(), x0, gamma)

# has_output_grad(::Half_Cauchy) = true
# has_argument_grads(::Half_Cauchy) = (true, true)

# export half_cauchy


In [12]:
# using  Gen

# @gen function a()
#     x ~ half_cauchy(0,1)

# end

# traces = [simulate(a, ()) for _ in 1:2]  # Run 100 importance samples

In [ ]:
using Gen
using Plots
using Statistics
using Distributions 




#my_dist = print(rand(half_cauchy(0,25)))
#my_dist = Truncated(Cauchy(), 0, Inf)


function compute_rhat(chains)
    m = length(chains)   # Number of chains
    n = length(chains[1])  # Number of samples per chain
   
    # Compute chain means
    chain_means = [mean(chain) for chain in chains]
    grand_mean = mean(chain_means)

    # Compute within-chain variance W
    W = mean([var(chain, corrected=true) for chain in chains])  # corrected=true uses n-1 in denominator
    #println("W", W)
    # Compute between-chain variance B
    B = (n / (m - 1)) * sum((chain_mean - grand_mean)^2 for chain_mean in chain_means)
    #println("B", B)
    # Compute potential scale reduction factor (R-hat)
    var_hat = ((n - 1) / n) * W + (B / n)
    r_hat = sqrt(var_hat / W)
    #println(var_hat)
    #println(r_hat)
    return r_hat
end





# Define the model with dynamic eta sampling
@gen function simple_normal_model(sigma)
    mu ~ normal(0, 1)     # Sample mu from a normal distribution
    tau  ~ normal(2, 5)   # Sample tau from a Half-Cauchy distribution
    
    # Dynamic eta values sampled from normal distributions
    list_of_Eta = [{(:eta, i)} ~ normal(0, 1) for i=1:length(sigma)]
    
    for i in 1:length(sigma)  # Loop over 5 iterations
        # Calculate theta based on mu, tau, and eta
        theta = mu + tau * list_of_Eta[i]
        
        # Sample obs from a normal distribution with mean theta and standard deviation sigma[i]
         {(:y, i)} ~ normal(theta, sigma[i])
    end

    
    
    
end

function do_inference(model, sigma, y_obs, num_iters, L, eps)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

    (trace, _) = generate(model, (sigma,), observations)
    accepted = 0
    mu_samples = []
    tau_samples = []

    # Store sampled values at each iteration
    for _ in 1:num_iters
        (trace, accepted_this_iter) = hmc(trace, select(:mu, :tau, (:eta, i for i in 1:length(sigma))...), L=L, eps=eps, check=true, observations=observations)
        #accepted += accepted_this_iter
        
        # Store samples
        final_choices = get_choices(trace)
        push!(mu_samples, final_choices[:mu])
        push!(tau_samples, final_choices[:tau])
    end

    #acceptance_rate = accepted / num_iters  # Compute acceptance rate

    return (mu_samples, tau_samples)
end



y_obs = [28, 8, -3, 7, -1, 1, 18, 12]
sigma =[15, 10, 16, 11, 9, 11, 10, 18]


# Use importance sampling as an example:
# traces = [simulate(simple_normal_model, (sigma,)) for _ in 1:100]  # Run 100 importance samples

# trace = do_inference(simple_normal_model, sigma, y_obs, 1000, 1, 0.1)

## Run inference



# acceptance_rates = []
# for (eps, L) in configurations
#     final_mu, final_tau, final_etas, accepted = do_inference(simple_normal_model, sigma, y_obs, 5000, L, eps)
#     println("Configuration: eps = $eps, L = $L")
#     println("Acceptance rate: $accepted")
#     #push!(acceptance_rates, (eps, L, acceptance_rate))
# end

# Run multiple chains
num_chains = 10
num_iters = 2000

configurations = [
    (i, j) for i in [0.01, 0.1, 0.5, 1, 2] for j in [1, 5, 10, 100, 1000]
]



for config in configurations
    chains_mu = []
    for _ in 1:num_chains
        mu_samples, _ = do_inference(simple_normal_model, sigma, y_obs, num_iters, config[2], config[1])
        push!(chains_mu, mu_samples)
    end
    println("(L, eps)", (config[2], config[1]))
    rhat_mu = compute_rhat(chains_mu)
    println("R-hat for mu: ", rhat_mu)
    means_per_chain = [mean(chain) for chain in chains_mu]
    println("Mean of mu for each chain: ", means_per_chain)
end

(L, eps)(1, 0.01)
R-hat for mu: 4.818024730049703
Mean of mu for each chain: [-0.37745695139405616, 0.9225521500816269, 1.0287125748969093, -0.6966661003774449, -0.03980640299113674, -0.7236675503770705, -1.250169370822809, 0.398405264872344, -1.8220888417309138, 0.6529145762240137]
(L, eps)(5, 0.01)
R-hat for mu: 1.430606578017749
Mean of mu for each chain: [0.08055376233555643, 0.7841014302378512, 1.7377485804069839, 0.12676361092631847, 0.4969624441849711, -0.09113419843261641, 0.7217038448853643, -0.6820842805996855, 0.8889451166938668, 1.4431513968610101]
(L, eps)(10, 0.01)
R-hat for mu: 1.0798300276397579
Mean of mu for each chain: [0.61789226163639, 0.8039627975541559, 0.5482347052667148, 0.054271038636818036, 0.3507273170507748, 0.13724758225102449, 0.5218186339050369, -0.45224937989851627, 0.5757792090620144, 0.3056655843168196]
(L, eps)(100, 0.01)
R-hat for mu: 1.0031490921239565
Mean of mu for each chain: [0.3694426453158926, 0.47023109601462953, 0.4889839493378542, 0.477755

In [2]:
using Gen
using Distributions
using LinearAlgebra

@gen function linear_regression_model(X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma)
    # Sample the prior for intercept alpha and regression coefficients beta
    alpha ~ normal(0, sqrt(sigma_alpha2))            # Intercept
    beta = [{(:beta, i)} ~ normal(mu_beta, sqrt(sigma_beta2)) for i in 1:K]      # Regression coefficients
    
    # Sample degrees of freedom and scale for the Student's t-distribution
    nu ~ gamma(2, 10)                                # Degrees of freedom for Student's t
    sigma ~ exponential(lambda_sigma)                 # Scale for the Student's t

    # Compute the mean for each y_i: μ_i = α + X_i * β
    for i in 1:N
       
        mu_i = alpha + dot(X[i, :], beta)           # Linear model: μ_i = α + X_i * β
        {(:y, i)} ~ student(mu_i, sigma)           # Sample y_i from Student's t distribution
    end
    
    return :y  # Return the observed responses
end


DynamicDSLFunction{Any}(Dict{Symbol, Any}(), Dict{Symbol, Any}(), Type[Any, Any, Any, Any, Any, Any, Any, Any], false, Union{Nothing, Some{Any}}[nothing, nothing, nothing, nothing, nothing, nothing, nothing, nothing], Main.var"##linear_regression_model#231", Bool[0, 0, 0, 0, 0, 0, 0, 0], false)

In [28]:
# # Example run function for inference (adjusted to your example)
# function do_inference_linear_regression(X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma, num_iters, L, eps)
#     # Set up observations as a choice map
#     observations = Gen.choicemap()
#     for (i, y) in enumerate(y)
#         observations[(:y, i)] = y
#     end

#     # Generate the model with initial data and priors
#     (trace, _) = generate(linear_regression_model, (X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma), observations)
    
#     accepted = 0
  
#     # Metropolis-Hastings on the parameters with HMC
#     for _ in 1:num_iters
#         # Run HMC with specified step size and trajectory length
      
#        (trace, accepted_this_iter) = hmc(trace, select(:alpha, :nu, :sigma,(:beta, i for i in 1:K)...), L=L, eps=eps, check = true, observations= observations)

#         accepted += accepted_this_iter
#     end

#     # Extract the final choices for parameters
#     final_choices = get_choices(trace)
#     final_alpha = final_choices[:alpha]
#     final_betas = [final_choices[(:beta, i)] for i in 1:K]
#     final_nu = final_choices[:nu]
#     final_sigma = final_choices[:sigma]

#     acceptance_rate = accepted / num_iters  # Divide by total number of proposals

#     return final_alpha, final_betas, final_nu, final_sigma, acceptance_rate
# end

# # Example of running inference with some random data
# N = 100  # Number of data points
# K = 5    # Number of features
# sigma_alpha2 = 1.0  # Prior variance for alpha
# mu_beta = 0.0  # Mean for beta prior
# sigma_beta2 = 1.0  # Variance for beta prior
# lambda_sigma = 1.0  # Rate for sigma prior

# # Generate synthetic data for testing
# X = randn(N, K)  # N x K feature matrix
# true_alpha = 2.0
# true_beta = randn(K)  # True regression coefficients
# nu_true = 5.0
# sigma_true = 1.0

# # Linear model for generating synthetic y values
# mu = X * true_beta .+ true_alpha
# y = randn(N) .* sigma_true .+ mu  # Add noise to y values

# # Run inference
# final_alpha, final_betas, final_nu, final_sigma, acceptance_rate = do_inference_linear_regression(
#     X, y, N, K, sigma_alpha2, mu_beta, sigma_beta2, lambda_sigma, 5000, 5 , 0.001
# )

# println("Final alpha: ", final_alpha)
# println("Final betas: ", final_betas)
# println("Final nu: ", final_nu)
# println("Final sigma: ", final_sigma)
# println("Acceptance rate: ", acceptance_rate)

Final alpha: 1.5000422579066275
Final betas: [1.1897398459134958, 0.9214583448843693, -0.1690514303478221, -0.31432676774621715, 0.8741174069727072]
Final nu: 0.9606564712034322
Final sigma: 2.057239020013213
Acceptance rate: 1.0


In [39]:


using Gen
using Plots
using Statistics
using Distributions 

# Define the model with dynamic eta sampling
@gen function eight_school_model(sigma)
    mu ~ normal(0, 1)     # Sample mu from a normal distribution
    tau  ~ normal(2, 5)   # Sample tau from a Half-Cauchy distribution
    
    # Dynamic eta values sampled from normal distributions
    list_of_Eta = [{(:eta, i)} ~ normal(0, 1) for i=1:length(sigma)]
    
    for i in 1:length(sigma)  # Loop over 5 iterations
        # Calculate theta based on mu, tau, and eta
        theta = mu + tau * list_of_Eta[i]
        
        # Sample obs from a normal distribution with mean theta and standard deviation sigma[i]
         {(:y, i)} ~ normal(theta, sigma[i])
    end

    
    
    
end

function multi_variable_metropolis(trace, model, sigma, observations, eps)
    # Extract current values of parameters
    mu_current = get_choices(trace)[:mu]
    tau_current = get_choices(trace)[:tau]
    eta_current = [get_choices(trace)[(:eta, i)] for i in 1:length(sigma)]
    
    
    # Propose new values for mu, tau, and eta
    mu_proposed = mu_current + eps * randn()
    tau_proposed = tau_current + eps * randn()
    eta_proposed = [eta_current[i] + eps * randn() for i in 1:length(sigma)]
    
     # Create a temporary choice map with the proposed values
    temp_cm = choicemap(
        (:mu => mu_proposed),
        (:tau => tau_proposed)
    )
   
    # Add proposed eta values to the choice map
    for i in 1:length(sigma)
        
        temp_cm[(:eta, i)] = eta_proposed[i]
        # print("hoi") 
    end

    # Use Gen.update to get the updated trace after applying the proposed values
    (proposed_trace, _, _) = Gen.update(
        trace,                # Current trace                # Generative model
        (sigma,), (),            # Arguments for the generative function
        temp_cm               # Temporary choice map with proposed values
    )

    ###
    current_score = get_score(trace)
    #print(current_score)
    
    proposed_score = get_score(proposed_trace)
    # print(proposed_score)
    # Compute the acceptance ratio using the gradients
    acceptance_ratio = min(1.0, exp(proposed_score - current_score))
    # print(acceptance_ratio)
    # Accept or reject based on the acceptance ratio
    if rand() < acceptance_ratio
        return (proposed_trace, 1)
    else
        return (trace, 0)  # Keep the current state if not accepted
    end
end

function do_inference(model, sigma, y_obs, num_iters, eps)
    observations = choicemap()
    for (i, y) in enumerate(y_obs)
        observations[(:y, i)] = y
    end

    (trace, _) = generate(model, (sigma,), observations)
    accepted = 0
    mu_samples = []
    tau_samples = []

    # Store sampled values at each iteration
    for _ in 1:num_iters
        (trace, accepted_this_iter) = multi_variable_metropolis(trace, model, sigma, observations, eps)
        #accepted += accepted_this_iter
        
        # Store samples
        final_choices = get_choices(trace)
        push!(mu_samples, final_choices[:mu])
        push!(tau_samples, final_choices[:tau])
    end

    #acceptance_rate = accepted / num_iters  # Compute acceptance rate

    return (mu_samples, tau_samples)
end





sigma = [15, 10, 16, 11, 9, 11, 10, 18]
y_obs = [28, 8, -3, 7, -1, 1, 18, 12]
num_iters = 1000  # Number of iterations for Metropolis-Hastings
eps = 0.1         # Step size for proposals

(mu_samples, tau_samples) = do_inference(eight_school_model, sigma, y_obs, num_iters, eps)
print(mean(mu_samples))



-0.8609198806009929